In [6]:
import pandas as pd

X_train = pd.read_parquet("../data/X_train_final.parquet")
y_train = pd.read_parquet("../data/y_train_final.parquet")

X_test = pd.read_parquet("../data/X_test_final.parquet")
y_test = pd.read_parquet("../data/y_test_final.parquet")

# Model building

I will build multiple models to evaluate their performance against one another so I can pick the best one for this usecase. I will start with a simple baseline model, and will choose Logistic Regression since we have a binary output. I will also start by using all features of the dataset to get a baseline model performance.

In [7]:
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train, y_train)

y_test_eval = y_test.squeeze()

y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test_eval, y_pred))
print("Precision:", precision_score(y_test_eval, y_pred))
print("Recall:", recall_score(y_test_eval, y_pred))
print("F1 Score:", f1_score(y_test_eval, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test_eval, y_pred))


Accuracy: 0.7381121362668559
Precision: 0.504302925989673
Recall: 0.7834224598930482
F1 Score: 0.6136125654450262
ROC AUC Score: 0.7525807951639154


/home/jaydanng/Projects/churn-prediction-system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


From these scores above, we can see that the model has fairly high recall of $0.783$ meaning that the model is pretty good at catching most of the churners. The downside of this is the low precision score of $0.504$ meaning the model also flags a lot of non churners as churners (false positives). Next we want to look into the features a little more to see what we can learn.

In [11]:
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": log_reg.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    by="abs_coefficient",
    ascending=False
)

coef_df.head(15)

,feature,coefficient,abs_coefficient
1,MonthlyCharges,-1.123589,1.123589
38,Contract_Two year,-0.781257,0.781257
16,InternetService_Fiber optic,0.705336,0.705336
2,TotalCharges,-0.663581,0.663581
36,Contract_Month-to-month,0.663399,0.663399
15,InternetService_DSL,-0.621009,0.621009
3,SeniorCitizen,0.457211,0.457211
35,StreamingMovies_Yes,0.274978,0.274978
25,DeviceProtection_No internet service,-0.274568,0.274568
17,InternetService_No,-0.274568,0.274568


From this we can see a few key insights:
1. First, we can see that the contract type is a big driver of churn. Customers with long term contracts are much less likely to churn compared to those with short term contracts (Two year vs Month to Month). 
2. Second, we can see that customers with fiber optic internet are much more likely to churn compared to DSL users.
3. Third, we can see the effect of tenure via the TotalCharges which shows once again that long term customers are much less likely to churn.
4. Finally, we see that senior citizens show a higher likelihood of churn

In this data, while monthly charges seems to be negatively associated with churn in the model, this is most likely due to interactions with other variables such as contract type and internet service. This just shows the importance of considering issues with multicollinearity when dealing with linear models.

The model identified contract type, internet service type, and customer tenure as the strongest predictors of churn. Customers on month-to-month contracts and those using fiber optic internet were significantly more likely to churn, while long-term contract holders and high-tenure customers showed strong retention. Additionally, senior customers exhibited higher churn risk. Some coefficients, such as monthly charges, reflect underlying correlations with other features, highlighting the impact of multicollinearity in linear models.

Next, we can try to make some slight improvements to this model by adjusting the threshold and also train some other types of models to see if they perform better than the logistic regression. I will try out a random forest model and a XGBoost model and compare the performance of all three.

In [12]:
import numpy as np

thresholds = np.arange(0.3, 0.8, 0.05)

for t in thresholds:
    y_pred_thresh = (y_proba >= t).astype(int)
    f1 = f1_score(y_test_eval, y_pred_thresh)
    print(f"Threshold: {t:.2f}, F1: {f1:.4f}")

Threshold: 0.30, F1: 0.5866
Threshold: 0.35, F1: 0.5995
Threshold: 0.40, F1: 0.6062
Threshold: 0.45, F1: 0.6125
Threshold: 0.50, F1: 0.6136
Threshold: 0.55, F1: 0.6176
Threshold: 0.60, F1: 0.6118
Threshold: 0.65, F1: 0.6125
Threshold: 0.70, F1: 0.6024
Threshold: 0.75, F1: 0.5732


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

/home/jaydanng/Projects/churn-prediction-system/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [14]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

In [ ]:
results = []

def evaluate_model(name, y_true, y_pred, y_proba):
    from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
    
    results.append({
        "Model": name,
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred)
    })

evaluate_model("LogReg", y_test_eval, y_pred, y_proba)
evaluate_model("RandomForest", y_test_eval, y_pred_rf, y_proba_rf)
evaluate_model("XGBoost", y_test_eval, y_pred_xgb, y_proba_xgb)

pd.DataFrame(results)

,Model,F1,ROC-AUC,Precision,Recall
0,LogReg,0.613613,0.841298,0.504303,0.783422
1,RandomForest,0.537764,0.821965,0.618056,0.475936
2,XGBoost,0.584203,0.842404,0.659933,0.524064


From this data above, we can see that the logistic regression model is still the best overall performer, but the XGBoost model is fairly class and is slighly more balanced. The RandomForest model is significantly worse than the other two so we can ignore it moving on. Next I will do some minor tuning to the XGBoost model to see if I can achieve better performance. 

In [ ]:
thresholds = np.arange(0.2, 0.7, 0.05)

for t in thresholds:
    y_pred_thresh = (y_proba_xgb >= t).astype(int)
    f1 = f1_score(y_test_eval, y_pred_thresh)
    print(f"Threshold: {t:.2f}, F1: {f1:.4f}")

Threshold: 0.20, F1: 0.6074
Threshold: 0.25, F1: 0.6173
Threshold: 0.30, F1: 0.6188
Threshold: 0.35, F1: 0.6194
Threshold: 0.40, F1: 0.6127
Threshold: 0.45, F1: 0.6073
Threshold: 0.50, F1: 0.5842
Threshold: 0.55, F1: 0.5487
Threshold: 0.60, F1: 0.5165
Threshold: 0.65, F1: 0.4498


In [21]:
best_threshold = 0.35

y_pred_xgb_best = (y_proba_xgb >= best_threshold).astype(int)

results.clear()

evaluate_model("LogReg", y_test_eval, y_pred, y_proba)
evaluate_model("RandomForest", y_test_eval, y_pred_rf, y_proba_rf)
evaluate_model("XGBoost", y_test_eval, y_pred_xgb_best, y_proba_xgb)

pd.DataFrame(results)

,Model,F1,ROC-AUC,Precision,Recall
0,LogReg,0.613613,0.841298,0.504303,0.783422
1,RandomForest,0.537764,0.821965,0.618056,0.475936
2,XGBoost,0.619385,0.842404,0.555085,0.700535


From this we can see that after adjusting the threshold for the XGBoost model, we have achieved our current best model. XGBoost achieved the best overall performance with an F1-score of 0.62 and ROC-AUC of 0.84 after threshold tuning, outperforming logistic regression by balancing precision and recall more effectively. Logistic regression, however, achieved higher recall (0.78), highlighting a tradeoff between identifying churners and minimizing false positives.

In [16]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_
}).sort_values(by="importance", ascending=False)

print(importance.head(10))

                           feature  importance
36         Contract_Month-to-month    0.382887
16     InternetService_Fiber optic    0.108009
18               OnlineSecurity_No    0.079691
27                  TechSupport_No    0.035374
38               Contract_Two year    0.026623
43  PaymentMethod_Electronic check    0.019975
35             StreamingMovies_Yes    0.019608
15             InternetService_DSL    0.019280
37               Contract_One year    0.017133
11                PhoneService_Yes    0.017106


Contract type is the most influential factor in predicting churn, with month-to-month customers significantly more likely to leave. Internet service type also plays a key role, as fiber optic users exhibit higher churn rates. Additionally, customers lacking value-added services such as online security and technical support show increased churn risk. Payment method and service usage patterns provide secondary signals, indicating behavioral differences between customers who stay and those who leave.